<a href="https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Answer

**One row = one content page (`content_id`)**, pseudonymized under one of 32 clients (`client_id`).
The grain check below confirms it: 30,000 rows, 30,000 unique `content_id`, zero duplicates — one row per page, not per page-day.

**Time window:** the performance metrics (impressions, clicks, sessions, `ctr`, `avg_position`, etc.) are aggregated over a **trailing 90-day window ending at export**.
Inside that window sit two narrower sub-windows used for trend comparison, `*_last_30d` and `*_prev_30d`.
Two other fields run on a *different*, longer clock: `content_age_days` (90–564 days) and `days_since_last_update` (1–373 days) describe the page's lifecycle, not the 90-day metrics window —
a page can be nearly two years old while only its most recent 90 days of search performance are measured here.

In [ ]:
import os
import pandas as pd

# Clone the repo if this is a fresh Colab session and the data isn't local yet
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ml-internship"):
        !git clone https://github.com/TheAlishbahWaheed/flyrank-ml-internship.git
    os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("Unique content_id:", df["content_id"].nunique(), "| duplicated rows:", df["content_id"].duplicated().sum())
print("Unique clients:", df["client_id"].nunique())
print("\ncontent_age_days range:", df["content_age_days"].min(), "-", df["content_age_days"].max())
print("days_since_last_update range:", df["days_since_last_update"].min(), "-", df["days_since_last_update"].max())

Shape: (30000, 44)
Unique content_id: 30000 | duplicated rows: 0
Unique clients: 32

content_age_days range: 90 - 564
days_since_last_update range: 1 - 373


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Answer

**Feature** — everything knowable about a page at the moment we'd score it for refresh priority: demand fields (`search_volume`, `competition*`, `cpc`), content fields (`content_type`, `main_intent`, `word_count`/`char_count` + tiers), the 90-day traffic/engagement counts and their `last_30d`/`prev_30d` splits, the lifecycle fields (`content_age_days`, `age_tier`, `freshness_tier`, `days_since_last_update`), and the current-state rate/tier fields (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`).

**Label / proxy** — `trend_direction` and `trend_pct`. Per the data dictionary, any declining-content label this project defines downstream is computed *from* `trend_direction`, so these two columns can never be features — using them would show the model its own answer.

**Context** — `content_id`, `client_id`: pseudonymous IDs, useful for grouping, joining, and a client-grouped train/test split — never as model inputs.

**Excluded** — `provider_used` (71.5% missing) and `model_used` (19.1% missing): internal content-generation tooling metadata, not a search/content signal, and too sparse to impute honestly. `age_tier_order`: a redundant integer re-encoding of `age_tier` with no new information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
feature_cols = [
    "search_volume","competition","competition_level","cpc","content_type","main_intent",
    "word_count","char_count","word_count_tier","char_count_tier",
    "impressions_90d","clicks_90d","pageviews_90d","sessions_90d","users_90d",
    "engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
    "days_with_impressions","days_with_sessions",
    "impressions_last_30d","clicks_last_30d","sessions_last_30d",
    "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d",
    "content_age_days","age_tier","freshness_tier","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct",
    "impression_tier","position_tier",
]

label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_id", "client_id"]
excluded_cols = {
    "provider_used": "71.5% missing, internal generation-pipeline metadata",
    "model_used": "19.1% missing, same reason as provider_used",
    "age_tier_order": "redundant integer encoding of age_tier",
}

all_cols = set(df.columns)
classified = set(feature_cols) | set(label_cols) | set(context_cols) | set(excluded_cols)
print("Columns accounted for:", len(classified), "/", len(all_cols))
print("Unclassified (should be empty):", all_cols - classified)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Columns accounted for: 44 / 44
Unclassified (should be empty): set()


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Answer

**Grain claim** — `content_id.duplicated().sum() == 0`, confirmed below: one row per page.

**Counts** — 30,000 rows across 32 clients; per-client row counts vary widely (see output) — this is not a balanced panel, some clients contribute far more pages than others.

**Missingness** — not random. `word_count`/`char_count` are 0% missing for `comparison article` and `feedly article`, but ~28% missing for `keyword article` — the largest `content_type` by far (27,207 of 30,000 rows). A blind `fillna(0)` would silently tell the model "this page has 0 words," which is wrong; a `has_word_count` flag is the honest fix.

**Windows** — `content_age_days` (90–564) and `days_since_last_update` (1–373) confirm pages are far older than the 90-day metrics window; history before that window is simply not observed, not zero.

**Rate columns** — `scroll_rate` and `ai_traffic_pct` both reach up to 300 in this data, confirming the dictionary's warning that these are not bounded 0–100% (numerator and denominator come from different measurement systems).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Grain
print("Duplicate content_id rows:", df["content_id"].duplicated().sum())

# Counts
per_client = df.groupby("client_id").size()
print("\nRows per client — min/median/max:", per_client.min(), per_client.median(), per_client.max())

# Missingness overall
print("\nTop missing columns (%):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(8))

# Missingness by content_type — is it patterned?
print("\nword_count missing % by content_type:")
print((df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean() * 100)).round(1))

# Windows
print("\ncontent_age_days:", df["content_age_days"].min(), "-", df["content_age_days"].max())
print("days_since_last_update:", df["days_since_last_update"].min(), "-", df["days_since_last_update"].max())

# Rate column sanity
for c in ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]:
    print(f"{c}: min={df[c].min()}, max={df[c].max()}")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Duplicate content_id rows: 0

Rows per client — min/median/max: 3 567.0 7008

Top missing columns (%):
provider_used        71.5
word_count           25.7
char_count           25.7
word_count_tier      25.7
char_count_tier      25.7
model_used           19.1
trend_pct            11.3
competition_level     8.7
dtype: float64

word_count missing % by content_type:
content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3
Name: word_count, dtype: float64

content_age_days: 90 - 564
days_since_last_update: 1 - 373
ctr: min=0.0, max=100.0
engagement_rate: min=0.0, max=100.0
scroll_rate: min=0.0, max=300.0
ai_traffic_pct: min=0.0, max=300.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Answer

- `avg_position == 0` (1,205 rows) means **no ranking data was recorded**, not literal rank zero. Treating it as position 0 would read as an implausibly strong ranking signal.
- `trend_pct` swings from -100% to +44,900%. The huge positive outliers come from tiny `*_prev_30d` denominators (a jump from ~1 impression to a few dozen reads as a multi-thousand-percent "trend"). It can't be used as a linear signal without capping or binning first.
- This is a **single 90-day cross-section**, one row per page — there is no repeated-observation history showing how a page's metrics moved *before* this window, so the analysis can only describe the current 90 days, not a real trajectory over the page's life.
- Missingness (`word_count`, `provider_used`, etc.) follows `content_type`, so any comparison across content types is partly a comparison of "has this metadata" vs. "doesn't" — a confound to flag, not fill away.
- IDs are pseudonymized with no way to tie a row back to a real client or page, so nothing here can be sanity-checked against outside context for a specific client.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# avg_position == 0 -> "no data", not rank zero
print("Rows with avg_position == 0:", (df["avg_position"] == 0).sum())

# trend_pct extreme range, and why
print("\ntrend_pct range:", df["trend_pct"].min(), "to", df["trend_pct"].max())
print(df.nlargest(3, "trend_pct")[["content_id", "impressions_prev_30d", "impressions_last_30d", "trend_pct"]])

# One row per page, confirmed again — no repeated observations over time
print("\nRows per content_id (should all be 1):")
print(df.groupby("content_id").size().value_counts())
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rows with avg_position == 0: 1205

trend_pct range: -100.0 to 44900.0
                 content_id  impressions_prev_30d  impressions_last_30d  \
24695  content_dd882c4152ac                     1                   450   
15405  content_a023517539fe                   757                212015   
14549  content_d020d42e7fcc                     9                  2364   

       trend_pct  
24695    44900.0  
15405    27907.3  
14549    26166.7  

Rows per content_id (should all be 1):
1    30000
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.